# CAR171 APT editor

Alejandro S. Borlaff, NASA ARC
Adapted from: Maxime Rizzo, NASA GSFC
Date: 5/10/26

In [1]:
import xml.etree.ElementTree as ET
import copy
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord

filename = 'CAR171.apt'
apt_file_dir = ''

CAR_dat = pd.read_csv("CAR171_apt_targets.csv")
CAR_dat
coords = SkyCoord(ra=CAR_dat['RA'].values*u.degree, dec=CAR_dat['DEC'].values*u.degree)

In [2]:
CAR_dat

,Unnamed: 0,Target,ArchiveTarget,description,category,RA,DEC,WFIPA_off,V3PA_off,V3PA_ori,...,RA_Source,Dec_Source,MPA_ThetaX,MPA_ThetaY,N_FRAME,N_EXP,N_FILT,Filter,MA Table,Resultant
0,0,SCA14_vertical_001,SCA14_vertical_001,Stray light test,Calibration,206.763704,49.027915,82.679537,142.679537,142.770046,...,206.885157,49.313267,0.2730,0.1149,55,2,1,F158,IM_171_10,10
1,1,SCA14_horizontal_002,SCA14_horizontal_002,Stray light test,Calibration,206.700223,49.091721,82.631251,142.631251,142.770046,...,206.885157,49.313267,0.2044,0.1480,55,2,1,F158,IM_171_10,10
2,2,SCA5_vertical_003,SCA5_vertical_003,Stray light test,Calibration,206.656449,49.567699,82.596225,142.596225,142.770046,...,206.885157,49.313267,-0.2713,0.1151,55,2,1,F158,IM_171_10,10
3,3,SCA5_horizontal_004,SCA5_horizontal_004,Stray light test,Calibration,206.619695,49.495314,82.568533,142.568533,142.770046,...,206.885157,49.313267,-0.2026,0.1481,55,2,1,F158,IM_171_10,10
4,4,SCA14_cen_005,SCA14_cen_005,Stray light test,Calibration,219.397404,53.825867,94.339101,154.339101,154.472328,...,219.563423,54.023340,0.2044,0.0823,55,2,1,F158,IM_171_10,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,77,SCAN_DRAGON2_078,SCAN_DRAGON2_078,Stray light test,Calibration,96.011116,-52.555319,92.337307,152.337307,152.355967,...,95.987958,-52.695661,-0.1408,-0.0083,55,4,1,F158,IM_193_11,11
78,78,SCAN_DRAGON2_079,SCAN_DRAGON2_079,Stray light test,Calibration,96.010741,-52.560915,92.337611,152.337611,152.355967,...,95.987958,-52.695661,-0.1352,-0.0083,55,4,1,F158,IM_193_11,11
79,79,SCAN_DRAGON2_080,SCAN_DRAGON2_080,Stray light test,Calibration,96.002663,-52.563516,92.344048,152.344048,152.355967,...,95.987958,-52.695661,-0.1324,-0.0035,55,4,1,F158,IM_193_11,11
80,80,NEGATIVE_GAP_1_081,NEGATIVE_GAP_1_081,Stray light test,Calibration,96.273100,-52.768972,92.128382,152.128382,152.355967,...,95.987958,-52.695661,0.0665,-0.1754,55,4,1,F158,IM_193_11,11


In [3]:
coords[0].ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)

np.str_('13 47 03.2889')

In [4]:

import xml.etree.ElementTree as ET

def update_target_coordinates(xml_path, output_path, new_coords):
    """
    Updates RA/Dec coordinates for each FixedTarget in the XML.

    Parameters:
        xml_path (str): Path to the input XML file.
        output_path (str): Path to write the updated XML file.
        new_coords (list of tuples): List of (RA, Dec) strings.
            Example: [("13 47 3.30", "+49 01 40.49"), ...]
            Must be same length as number of FixedTarget entries.
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # XML uses namespaces; must extract them for searching.
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Find all FixedTarget entries
    targets = root.findall('.//ns:FixedTarget', ns)

    if len(new_coords) != len(targets):
        raise ValueError(
            f"Provided {len(new_coords)} coordinates but XML contains {len(targets)} FixedTargets."
        )

    for (target, coord) in zip(targets, new_coords):
        ra = coord.ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)
        dec = coord.dec.to_string(unit=u.degree, sep=' ', precision=2, pad=True, alwayssign=True)
        # Find EquatorialCoordinates node
        eq = target.find('ns:EquatorialCoordinates', ns)
        if eq is not None:
            # Format must match XML's "Value" attribute: "RA Dec"
            eq.set("Value", f"{ra} {dec}")
            # print("Value", f"{ra} {dec}")
        else:
            print(f"Warning: FixedTarget missing EquatorialCoordinates element.")

    # Save modified XML
    tree.write(output_path, encoding="UTF-8", xml_declaration=True)
    print(f"Updated file written to: {output_path}")



def sync_passplan_numbers(xml_input, xml_output):
    """
    Updates each <PassPlan Number="X"> so that X matches its TargetSelection Fixed target ID.
    Example:
        <TargetSelection>Fixed: 13</TargetSelection>
        → PassPlan Number becomes "13".
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Namespace used by Roman APT XML
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Iterate through all PassPlan entries
    for passplan in root.findall('.//ns:PassPlan', ns):
        ts = passplan.find('ns:TargetSelection', ns)
        if ts is not None and "Fixed:" in ts.text:
            # Extract target number from "Fixed: N"
            target_num = ts.text.split("Fixed:")[1].strip()
            passplan.set("Number", target_num)

    # Save output
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"Updated XML saved to {xml_output}")



def sort_surveyplan_steps(xml_input, xml_output):
    """
    Sorts all <SurveyPlanStep> entries by their <PassPlan> value in increasing order.
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Extract namespace automatically
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Locate the <SurveyPlan> container
    survey_plan = root.find('.//ns:SurveyPlan', ns)
    if survey_plan is None:
        raise RuntimeError("Could not find <SurveyPlan> in XML.")

    # Extract all SurveyPlanStep elements
    steps = survey_plan.findall('ns:SurveyPlanStep', ns)

    # Sort by numeric PassPlan value
    def get_passplan_number(step):
        pp = step.find('ns:PassPlan', ns)
        return int(pp.text.strip()) if pp is not None else 999999999

    steps_sorted = sorted(steps, key=get_passplan_number)

    # Clear existing order
    for step in steps:
        survey_plan.remove(step)

    # Reinsert in sorted order
    for step in steps_sorted:
        survey_plan.append(step)

    # Save output
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"SurveyPlan sorted and saved to {xml_output}")



def update_orient_ranges(xml_input, xml_output, orient_min_list, orient_max_list):
    """
    Updates OrientRange OrientMin/OrientMax in each SurveyPlanStep using
    user-provided arrays.
    
    orient_min_list and orient_max_list must have the same length as the
    number of SurveyPlanStep entries.
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Extract namespace automatically
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Find all SurveyPlanStep entries
    steps = root.findall('.//ns:SurveyPlan/ns:SurveyPlanStep', ns)

    if len(steps) != len(orient_min_list) or len(steps) != len(orient_max_list):
        raise ValueError("ERROR: Input arrays must match number of SurveyPlanStep entries.")

    # Update OrientMin and OrientMax for each step
    for step, new_min, new_max in zip(steps, orient_min_list, orient_max_list):
        orient_range = step.find('ns:SpecialRequirements/ns:OrientRange', ns)
        if orient_range is not None:
            orient_range.set("OrientMin", f"{new_min} Degrees")
            orient_range.set("OrientMax", f"{new_max} Degrees")
        else:
            print(f"Warning: No OrientRange found in SurveyPlanStep uid={step.get('uid')}")

    # Save updated XML
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"Updated XML saved to {xml_output}")


# Example usage:
# new_min = [142.1, 143.2, 144.3, ...]
# new_max = [142.1, 143.2, 144.3, ...]


In [5]:
sync_passplan_numbers(xml_input="CAR171.apt", xml_output="CAR171_sync.apt")
sort_surveyplan_steps(xml_input="CAR171_sync.apt", xml_output="CAR171_order.apt")
update_target_coordinates(xml_path="CAR171_order.apt", output_path="CAR171_mod.apt", new_coords=coords)
update_orient_ranges(xml_input="CAR171_mod.apt", xml_output="CAR171_orient.apt", orient_min_list=CAR_dat["V3PA_off"], orient_max_list=CAR_dat["V3PA_off"])

Updated XML saved to CAR171_sync.apt
SurveyPlan sorted and saved to CAR171_order.apt
Updated file written to: CAR171_mod.apt
Updated XML saved to CAR171_orient.apt


In [6]:
coords

<SkyCoord (ICRS): (ra, dec) in deg
    [(206.76370388,  49.02791529), (206.70022344,  49.09172074),
     (206.65644858,  49.56769873), (206.6196955 ,  49.49531373),
     (219.39740381,  53.82586732), (219.4497494 ,  54.23180275),
     (206.76370388,  49.02791529), (206.70022344,  49.09172074),
     (206.65644858,  49.56769873), (206.6196955 ,  49.49531373),
     (219.39740381,  53.82586732), (219.4497494 ,  54.23180275),
     ( 88.23938768, -48.53277318), ( 86.31433203, -55.78961179),
     ( 93.20664533, -58.65779843), (104.68455933, -55.02936937),
     (104.06736775, -50.17161332), ( 94.64784295, -46.59317013),
     ( 86.50627521, -50.07364142), ( 85.36270082, -53.97538033),
     ( 94.47369094, -51.3459329 ), ( 94.19902125, -53.93989752),
     ( 96.04397497, -52.62818875), ( 96.03480378, -52.76567658),
     ( 96.00303999, -52.55792091), ( 95.98650488, -52.83374204),
     ( 88.71597612, -48.07242579), ( 88.50784436, -48.22204991),
     ( 88.2997889 , -48.3703499 ), ( 88.08928918, -48.5